# Voice Guard demo - reproducible launcher

Run the cells from top to bottom. The launcher defaults to the pushed `main` branch and records the exact resolved commit SHA. Set the optional `VOICE_GUARD_REF` environment variable to pin another branch, tag, or commit. Store ngrok authentication in the Kaggle secret `NGROK_AUTHTOKEN`; this notebook never contains or prints the token.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Shruti-sivaramakrishnan/indic-fake-voice-detection.git"
REPO_DIR = Path("/kaggle/working/indic-fake-voice-detection")
VOICE_GUARD_REF = os.environ.get("VOICE_GUARD_REF", "main").strip() or "main"

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", VOICE_GUARD_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-kaggle.txt")], check=True)
RESOLVED_SHA = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", RESOLVED_SHA)


The launcher below uses the Kaggle secret `NGROK_AUTHTOKEN` when present, otherwise it reuses any ngrok authentication already configured in the session. It creates a random public URL to avoid static-endpoint conflicts.

In [ ]:
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

from pyngrok import ngrok

try:
    from kaggle_secrets import UserSecretsClient
    ngrok_token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
except Exception:
    ngrok_token = None
if ngrok_token:
    ngrok.set_auth_token(ngrok_token)

PORT = 8501
LOG_PATH = Path("/kaggle/working/voice_guard_streamlit.log")
log_stream = LOG_PATH.open("w", encoding="utf-8")
server = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py", "--server.address", "0.0.0.0", "--server.port", str(PORT), "--server.headless", "true"],
    cwd=REPO_DIR,
    stdout=log_stream,
    stderr=subprocess.STDOUT,
)

def log_tail(lines=80):
    log_stream.flush()
    content = LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()
    return "\n".join(content[-lines:])

tunnel = None
try:
    health_url = f"http://127.0.0.1:{PORT}/_stcore/health"
    deadline = time.time() + 120
    while time.time() < deadline:
        if server.poll() is not None:
            raise RuntimeError(f"Streamlit exited early.\n{log_tail()}")
        try:
            with urllib.request.urlopen(health_url, timeout=3) as response:
                if response.status == 200:
                    break
        except Exception:
            time.sleep(2)
    else:
        raise TimeoutError(f"Streamlit health check timed out.\n{log_tail()}")

    ngrok.kill()
    tunnel = ngrok.connect(addr=PORT, proto="http", bind_tls=True)
    print("Voice Guard URL:", tunnel.public_url)
    print("This cell must keep running while the URL is in use.")
    while server.poll() is None:
        time.sleep(2)
    raise RuntimeError(f"Streamlit stopped.\n{log_tail()}")
except KeyboardInterrupt:
    print("Stopping Voice Guard...")
except Exception:
    print(log_tail())
    raise
finally:
    if tunnel is not None:
        try:
            ngrok.disconnect(tunnel.public_url)
        except Exception:
            pass
    try:
        ngrok.kill()
    except Exception:
        pass
    if server.poll() is None:
        server.terminate()
        try:
            server.wait(timeout=10)
        except subprocess.TimeoutExpired:
            server.kill()
    log_stream.close()
